In [2]:
import os
import random
import cv2
import numpy as np
from pathlib import Path
import shutil

In [ ]:
def adjust_brightness_contrast(image, brightness_factor=None, contrast_factor=None):
    """
    Adjust brightness and contrast of an image
    brightness_factor: if > 1, increases brightness. if < 1, decreases brightness
    contrast_factor: if > 1, increases contrast. if < 1, decreases contrast
    """
    if brightness_factor is None:
        # Random brightness factor between 0.6 and 1.4
        brightness_factor = random.uniform(0.6, 1.4)

    if contrast_factor is None:
        # Random contrast factor between 0.7 and 1.3
        contrast_factor = random.uniform(0.7, 1.3)

    # Adjust brightness
    bright_img = cv2.multiply(image, brightness_factor)

    # Adjust contrast
    mean = np.mean(bright_img)
    contrast_img = cv2.addWeighted(bright_img, contrast_factor, mean, 0, 0)

    # Clip values to valid range [0, 255]
    final_img = np.clip(contrast_img, 0, 255).astype(np.uint8)

    return final_img, brightness_factor, contrast_factor

In [ ]:
def process_image_and_annotation_brightness(image_path, label_path, output_image_path, output_label_path):
    """Process image by changing its exposure/brightness and copy its annotation"""
    # Read image
    img = cv2.imread(str(image_path))

    # Adjust brightness and contrast
    adjusted_img, brightness, contrast = adjust_brightness_contrast(img)

    # Copy annotation file if it exists (no modification needed for brightness changes)
    if os.path.exists(label_path):
        shutil.copy2(label_path, output_label_path)

    # Save adjusted image
    cv2.imwrite(str(output_image_path), adjusted_img)

    return brightness, contrast

In [ ]:
def rotate_point(x, y, angle, cx=0.5, cy=0.5):
    """Rotate a point around a center point
    x, y: normalized coordinates (0-1)
    angle: angle in degrees
    cx, cy: center point (default is image center 0.5, 0.5)
    """
    # Convert angle to radians
    angle_rad = np.radians(angle)

    # Translate point to origin
    x_shifted = x - cx
    y_shifted = y - cy

    # Rotate point
    x_rotated = x_shifted * np.cos(angle_rad) - y_shifted * np.sin(angle_rad)
    y_rotated = x_shifted * np.sin(angle_rad) + y_shifted * np.cos(angle_rad)

    # Translate back
    x_final = x_rotated + cx
    y_final = y_rotated + cy

    return x_final, y_final

In [ ]:
def rotate_yolo_annotation(annotation, angle):
    """Rotate YOLO format annotation
    annotation: list containing [class_id, x_center, y_center, width, height]
    angle: rotation angle in degrees
    """
    class_id = annotation[0]
    x_center, y_center = rotate_point(annotation[1], annotation[2], angle)

    # For width and height, we don't rotate them as they are dimensions
    # but for angles that are multiples of 90 degrees, we swap them
    width = annotation[3]
    height = annotation[4]

    if angle in [90, 270]:
        width, height = height, width

    # Ensure coordinates stay within bounds [0, 1]
    x_center = np.clip(x_center, 0, 1)
    y_center = np.clip(y_center, 0, 1)

    return [class_id, x_center, y_center, width, height]

In [ ]:
def process_image_and_annotation_rotate(image_path, label_path, output_image_path, output_label_path, angle):
    """Process both image and its annotation file"""
    # Read and rotate image
    img = cv2.imread(str(image_path))
    height, width = img.shape[:2]
    center = (width // 2, height // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

    # Process annotation file if it exists
    if os.path.exists(label_path):
        print("path exist");
        with open(label_path, 'r') as f:
            annotations = f.readlines()

        rotated_annotations = []
        for ann in annotations:
            # Convert string annotation to list of floats
            ann_parts = list(map(float, ann.strip().split()))
            # Rotate annotation
            rotated_ann = rotate_yolo_annotation(ann_parts, angle)
            # Convert back to string format
            rotated_annotations.append(' '.join(map(str, rotated_ann)))

        # Write rotated annotations
        with open(output_label_path, 'w') as f:
            f.write('\n'.join(rotated_annotations))

    # Save rotated image
    cv2.imwrite(str(output_image_path), rotated_img)

In [8]:
def ajouter_brouillard(image_or_path, intensite=0.5):
    """
    Ajoute un effet de brouillard sur une image.
    image_or_path : chemin (str / PathLike) OU numpy.ndarray (BGR uint8)
    intensite : float entre 0 et 1
    Retour : image_brouillard (numpy.uint8 BGR)
    """
    # Charger si on a reçu un chemin
    if isinstance(image_or_path, (str, os.PathLike)):
        image = cv2.imread(str(image_or_path))
        if image is None:
            raise FileNotFoundError(f"Impossible de charger l'image : {image_or_path}")
    elif isinstance(image_or_path, np.ndarray):
        image = image_or_path.copy()
    else:
        raise TypeError("image_or_path doit être un chemin (str/PathLike) ou un numpy.ndarray")

    # Normaliser l'image (valeurs entre 0 et 1)
    image = image.astype(np.float32) / 255.0

    # Générer un nuage de brouillard (bruit flou)
    hauteur, largeur = image.shape[:2]
    bruit = np.random.normal(loc=0.5, scale=0.5, size=(hauteur, largeur)).astype(np.float32)
    brouillard = cv2.GaussianBlur(bruit, (0, 0), sigmaX=max(1.0, hauteur/10), sigmaY=max(1.0, largeur/10))

    # Normaliser et étendre sur 3 canaux
    brouillard = cv2.normalize(brouillard, None, 0, 1, cv2.NORM_MINMAX)
    # Ajouter une dimension pour avoir (hauteur, largeur, 1), puis répéter
    brouillard = brouillard[:, :, np.newaxis]
    brouillard = np.repeat(brouillard, 3, axis=2)

    # Mélange linéaire entre image originale et brouillard
    image_brouillard = cv2.addWeighted(image, 1 - float(intensite), brouillard, float(intensite), 0)

    # Revenir à l'échelle 0—255
    image_brouillard = (np.clip(image_brouillard, 0.0, 1.0) * 255).astype(np.uint8)

    return image_brouillard

In [ ]:
import cv2
import numpy as np
import os
import random
import shutil
def traiter_images_fog(train_dir, output_dir, labels_dir=None, output_labels_dir=None,
                          n_images=250, intensite_max=0.5, intensite_min=0.3):
    """
    Sélectionne n_images aléatoires dans train_dir, applique le brouillard,
    et sauvegarde dans output_dir. Duplique également les labels correspondants.

    Args:
        train_dir: Dossier contenant les images d'entraînement
        output_dir: Dossier de sortie pour les images avec brouillard
        labels_dir: Dossier contenant les labels (fichiers .txt)
        output_labels_dir: Dossier de sortie pour les labels dupliqués
        n_images: Nombre d'images à traiter
        intensite_max: Intensité maximale du brouillard
        intensite_min: Intensité minimale du brouillard
    """
    # Créer les dossiers de sortie
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    if output_labels_dir and not os.path.exists(output_labels_dir):
        os.makedirs(output_labels_dir)

    # Lister toutes les images
    toutes_images = [f for f in os.listdir(train_dir)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

    if len(toutes_images) == 0:
        print("⚠️ Aucun fichier image trouvé dans le dossier.")
        return

    # Sélection aléatoire
    images_selectionnees = random.sample(toutes_images, min(n_images, len(toutes_images)))

    labels_copies = 0
    images_traitees = 0

    for nom_fichier in images_selectionnees:
        chemin_entree = os.path.join(train_dir, nom_fichier)

        # Traiter l'image
        image = cv2.imread(chemin_entree)
        if image is None:
            print(f"⚠️ Impossible de lire {nom_fichier}")
            continue

        intensite = random.uniform(intensite_min, intensite_max)
        image_modifiee = ajouter_brouillard(image, intensite=intensite)

        nouveau_nom = f"fog_{nom_fichier}"
        chemin_sortie = os.path.join(output_dir, nouveau_nom)
        cv2.imwrite(chemin_sortie, image_modifiee)
        images_traitees += 1
        print(f"✅ Image traitée : {nouveau_nom}")

        # Copier le label correspondant si demandé
        if labels_dir and output_labels_dir:
            # Trouver le fichier label correspondant (même nom mais .txt)
            nom_base = os.path.splitext(nom_fichier)[0]
            nom_label = f"{nom_base}.txt"
            chemin_label = os.path.join(labels_dir, nom_label)

            if os.path.exists(chemin_label):
                nouveau_nom_label = f"fog_{nom_label}"
                chemin_label_sortie = os.path.join(output_labels_dir, nouveau_nom_label)
                shutil.copy2(chemin_label, chemin_label_sortie)
                labels_copies += 1
                print(f"   📄 Label copié : {nouveau_nom_label}")
            else:
                print(f"   ⚠️ Label non trouvé : {nom_label}")

    print(f"\n✅ Traitement terminé !")
    print(f"   - {images_traitees} images modifiées dans : {output_dir}")
    if output_labels_dir:
        print(f"   - {labels_copies} labels copiés dans : {output_labels_dir}")


In [ ]:
if __name__ == "__main__":
    dossier_train_images = "./dataset/images/train"
    dossier_prepare = "./images/train_prepare/"
    dossier_labels="./dataset/labels/train"
    dossier_labels_sortie="./labels/train_prepare"
    traiter_images_fog(
        train_dir=dossier_train_images,
        output_dir=dossier_prepare,
        labels_dir=dossier_labels,
        output_labels_dir=dossier_labels_sortie,
        n_images=10,
        intensite_max=0.7,
        intensite_min=0.2
    )
    selected_images = random.sample(dossier_train_images, min(200, len(dossier_train_images)))
    print(selected_images);

    # Possible rotation angles
    angles = [90, 180, 270]

    # Process each selected image
    for img_path in selected_images:
        # Get corresponding label path
        label_path = dossier_labels / f"{img_path.stem}.txt"

        # Randomly select an angle
        angle = random.choice(angles)

        # Create output paths
        output_image_path = dossier_prepare / f"{img_path.stem}_rot{angle}.jpg"
        output_label_path = dossier_labels_sortie / f"{img_path.stem}_rot{angle}.txt"

        # Process the image and its annotation
        process_image_and_annotation(
            img_path,
            label_path,
            output_image_path,
            output_label_path,
            angle
        )
        print(f"Processed {img_path.name} with {angle}° rotation")
    process_image_and_annotation_brightness(dossier_train_images, dossier_train_label, dossier_prepare, dossier_labels_sortie)




✅ Image traitée : fog_gss1417_jpg.rf.975d8860060c4ab7bf90e1ee42930c08.jpg
   📄 Label copié : fog_gss1417_jpg.rf.975d8860060c4ab7bf90e1ee42930c08.txt
✅ Image traitée : fog_gss719_jpg.rf.11fa4752b606d61bf9861adf11e27b96.jpg
   📄 Label copié : fog_gss719_jpg.rf.11fa4752b606d61bf9861adf11e27b96.txt
✅ Image traitée : fog_gss1456_jpg.rf.a2340666e29e8b5548715c6361c20fae.jpg
   📄 Label copié : fog_gss1456_jpg.rf.a2340666e29e8b5548715c6361c20fae.txt
✅ Image traitée : fog_gss750_jpg.rf.8e1a222530a932a987a41730d5669def.jpg
   📄 Label copié : fog_gss750_jpg.rf.8e1a222530a932a987a41730d5669def.txt
✅ Image traitée : fog_gss482_jpg.rf.565356f0eba214222d30286fca7be89b.jpg
   📄 Label copié : fog_gss482_jpg.rf.565356f0eba214222d30286fca7be89b.txt
✅ Image traitée : fog_gss1623_jpg.rf.8cbe468c50ab564c1c4e8526bd046358.jpg
   📄 Label copié : fog_gss1623_jpg.rf.8cbe468c50ab564c1c4e8526bd046358.txt
✅ Image traitée : fog_gss65_jpg.rf.57007184d0e04d9718dc73616d68cf6b.jpg
   📄 Label copié : fog_gss65_jpg.rf.5700

AttributeError: 'str' object has no attribute 'stem'

In [22]:
import shutil
import os

# Chemin du dossier à supprimer
dossier_a_supprimer = "./images"

# Vérifier si le dossier existe
if os.path.exists(dossier_a_supprimer):
    # Supprimer le dossier et tout son contenu
    shutil.rmtree(dossier_a_supprimer)
    print(f"✅ Dossier '{dossier_a_supprimer}' supprimé avec succès !")
else:
    print(f"⚠️ Le dossier '{dossier_a_supprimer}' n'existe pas.")

✅ Dossier './images' supprimé avec succès !
